In [19]:
import re
import pandas as pd
from playwright.async_api import async_playwright
import json
from pathlib import Path
import json
import urllib.parse
import urllib.request
from http.cookiejar import CookieJar
from pathlib import Path

In [7]:
data = json.loads(Path("geography.json").read_text(encoding="utf-8"))

In [16]:
data

{'generated_at': '2026-03-12T14:05:23.268247+00:00',
 'source': 'https://www.fuelprices.gr/GetGeography?nomos=<nomos_code>',
 'nomoi': {'A1000000': {'name': 'ΝΟΜΑΡΧΙΑ ΑΘΗΝΩΝ',
   'geographies': [{'code': 'A1020000',
     'name': 'ΔΗΜΟΣ ΑΓΙΑΣ ΒΑΡΒΑΡΑΣ',
     'kind': 'ΔΗΜΟΣ',
     'children': [{'code': 'A1020100',
       'name': 'Δ.Δ.Αγίας Βαρβάρας',
       'kind': 'Δ.Δ.Αγίας'}]},
    {'code': 'A1030000',
     'name': 'ΔΗΜΟΣ ΑΓΙΑΣ ΠΑΡΑΣΚΕΥΗΣ',
     'kind': 'ΔΗΜΟΣ',
     'children': [{'code': 'A1030100',
       'name': 'Δ.Δ.Αγίας Παρασκευής',
       'kind': 'Δ.Δ.Αγίας'}]},
    {'code': 'A1040000',
     'name': 'ΔΗΜΟΣ ΑΓΙΟΥ ΔΗΜΗΤΡΙΟΥ',
     'kind': 'ΔΗΜΟΣ',
     'children': [{'code': 'A1040100',
       'name': 'Δ.Δ.Αγίου Δημητρίου',
       'kind': 'Δ.Δ.Αγίου'}]},
    {'code': 'A1050000',
     'name': 'ΔΗΜΟΣ ΑΓΙΩΝ ΑΝΑΡΓΥΡΩΝ',
     'kind': 'ΔΗΜΟΣ',
     'children': [{'code': 'A1050100',
       'name': 'Δ.Δ.Αγίων Αναργύρων',
       'kind': 'Δ.Δ.Αγίων'}]},
    {'code': 'A1010000',
     'name':

In [15]:
nomoi = list(data['nomoi'].keys())
nomoi[0]

'A1000000'

In [50]:
BASE_URL = "https://www.fuelprices.gr/GetGeography"

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_8) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/145.0.0.0 Safari/537.36"
    ),
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9,el;q=0.8",
}

In [54]:
def load_dimos_codes(geography_path: str, nomos_code: str) -> list[str]:
    data = json.loads(Path(geography_path).read_text(encoding="utf-8"))
    return [item["code"] for item in data["nomoi"][nomos_code]["geographies"]]


def fetch_geography_results(nomos_code: str, geography_path: str = "geography.json") -> str:
    dimos_codes = load_dimos_codes(geography_path, nomos_code)

    cookie_jar = CookieJar()
    opener = urllib.request.build_opener(urllib.request.HTTPCookieProcessor(cookie_jar))

    initial_url = f"{BASE_URL}?{urllib.parse.urlencode({'nomos': nomos_code})}"
    initial_req = urllib.request.Request(initial_url, headers=HEADERS)
    with opener.open(initial_req, timeout=30):
        pass

    params = [
        ("nomos", nomos_code),
        ("return_to", "CheckPrices"),
        *[("dimos", code) for code in dimos_codes],
        ("submit", "Επόμενο"),
    ]

    url = f"{BASE_URL}?{urllib.parse.urlencode(params)}"
    req = urllib.request.Request(
        url,
        headers={**HEADERS, "Referer": initial_url},
    )
    return url


In [55]:
url = fetch_geography_results(nomoi[0])

In [57]:
playwright = await async_playwright().start()
browser = await playwright.chromium.launch(headless=False)

# Create a new browser window
context = await browser.new_context()
page = await context.new_page()

# Tell it to go to this page
await page.goto(url)
await page.locator("input.button", has_text="Επόμενο").click()
await page.locator('input[name="prodclass"][value="1"]').click()
await page.frame_locator("iframe[title='reCAPTCHA']").locator(".recaptcha-checkbox-border").click()
await context.storage_state(path="state.json")
await page.wait_for_timeout(8000)
await page.locator('input[type="submit"][value="Επόμενο"]').click()

TimeoutError: Locator.click: Timeout 30000ms exceeded.
Call log:
  - waiting for locator("input[type=\"submit\"][value=\"Επόμενο\"]")
    - locator resolved to <input name="submit" type="submit" class="button" value="Επόμενο"/>
  - attempting click action
    2 × waiting for element to be visible, enabled and stable
      - element is visible, enabled and stable
      - scrolling into view if needed
      - done scrolling
      - <div></div> from <div>…</div> subtree intercepts pointer events
    - retrying click action
    - waiting 20ms
    2 × waiting for element to be visible, enabled and stable
      - element is visible, enabled and stable
      - scrolling into view if needed
      - done scrolling
      - <div></div> from <div>…</div> subtree intercepts pointer events
    - retrying click action
      - waiting 100ms
    58 × waiting for element to be visible, enabled and stable
       - element is visible, enabled and stable
       - scrolling into view if needed
       - done